In [1]:
from pymongo import MongoClient
import pandas as pd

## UTILS FOR READING MONGO


In [2]:
def _connect_mongo(host, port, username, password, db):
    """A util for making a connection to mongo"""

    if username and password:
        mongo_uri = "mongodb://%s:%s@%s:%s/%s" % (
            username, password, host, port, db)
        conn = MongoClient(mongo_uri)
    else:
        conn = MongoClient(host, port)

    return conn[db]


def read_mongo(
    db,
    collection,
    query={},
    host="localhost",
    port=27017,
    username=None,
    password=None,
    no_id=True,
):
    """Read from Mongo and Store into DataFrame"""

    # Connect to MongoDB
    db = _connect_mongo(
        host=host, port=port, username=username, password=password, db=db
    )

    # Make a query to the specific DB and Collection
    cursor = db[collection].find(query)

    # Expand the cursor and construct the DataFrame
    df = pd.DataFrame(list(cursor))

    # Delete the _id
    if no_id:
        del df["_id"]

    return df

## Init db and df


In [3]:
config = pd.read_json("./config.json")
dbConfig = config["Database"]
dbConfig

DBNAME         statistic_service
HOST                27.74.255.96
PASS                    12345678
PORT                       28118
USER      statistic_service_user
Name: Database, dtype: object

In [4]:
df = read_mongo(
    dbConfig["DBNAME"],
    "cesti_online",
    {},
    dbConfig["HOST"],
    dbConfig["PORT"],
    dbConfig["USER"],
    dbConfig["PASS"],
)
df.columns.values

array(['newsId', 'categoryId', 'topicId', 'mainCategoryId', 'title',
       'topicName', 'categoryName', 'mainCategoryName', 'newsDate',
       'topicDate'], dtype=object)

In [5]:
# df["newsDate"] = pd.to_datetime(df["newsDate"], format="ISO8601", utc=True)
# df["topicDate"] = pd.to_datetime(df["topicDate"], format="ISO8601", utc=True)
df.dtypes

newsId                      object
categoryId                  object
topicId                     object
mainCategoryId              object
title                       object
topicName                   object
categoryName                object
mainCategoryName            object
newsDate            datetime64[ns]
topicDate           datetime64[ns]
dtype: object

## Filter count


In [6]:
filters = ((df["newsDate"] >= pd.to_datetime("2023-10-19 00:00:00"))) & (
    df["newsDate"] <= pd.to_datetime("2023-10-19 23:59:59")
)

# octo = df.query('`newsDate`>="2023-01-01T00:00:00Z" AND `newsDate`<="2023-11-19T17:58:03Z"')

In [7]:
df[filters].groupby(["mainCategoryId", "mainCategoryName"])[
    ["topicId", "newsId"]
].nunique().sort_values(["topicId", "newsId"], ascending=False)

,,topicId,newsId
mainCategoryId,mainCategoryName,,
de5cc5b1-1bca-4f05-90fc-59024e4172e6,Hoạt đông nghiên cứu khoa học,4,9
024a3964-d072-45e3-bb7a-67a5c263fbe6,Hoạt động thị trường khoa học và công nghệ,1,1
29c7f3a3-a950-4efe-9a29-8b57a16152b5,Hoạt động đổi mới sáng tạo và khởi nghiệp đổi mới sáng tạo,1,1


## Filter count over time


In [83]:
# df[(df["mainCategoryName"] == "Hoạt đông nghiên cứu khoa học")].groupby(
#     ["mainCategoryName", pd.Grouper(key="newsDate", freq="h")]
# )["newsId"].count()
filters = (
    ((df["topicDate"] >= pd.to_datetime("2023-10-1 00:00:00")))
    & (df["topicDate"] <= pd.to_datetime("2023-10-31 23:59:59"))
    & (df["mainCategoryName"] == "Hoạt đông nghiên cứu khoa học")
)

df_date = pd.DataFrame(
    df[filters]
    .groupby(["mainCategoryName", pd.Grouper(key="topicDate", freq="D")])["topicId"]
    .nunique()
)
df_date.reset_index().set_index("topicDate").groupby(["mainCategoryName"])[
    "topicId"
].resample("D").asfreq().fillna(0)

mainCategoryName               topicDate 
Hoạt đông nghiên cứu khoa học  2023-10-02     3.0
                               2023-10-03    31.0
                               2023-10-04     2.0
                               2023-10-05     1.0
                               2023-10-06     1.0
                               2023-10-07     0.0
                               2023-10-08     3.0
                               2023-10-09     5.0
                               2023-10-10     0.0
                               2023-10-11     1.0
                               2023-10-12     2.0
                               2023-10-13     3.0
                               2023-10-14     0.0
                               2023-10-15     5.0
                               2023-10-16     1.0
                               2023-10-17     3.0
                               2023-10-18     2.0
                               2023-10-19     2.0
                               2023-10-20     4.0
        